## 1. Configure Spark and Load Gold Layer Data
Set up PySpark with Hudi support and load all Gold layer tables for fraud typology analysis.

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf

spark_jars = os.environ.get("SPARK_JARS", "")
jar_list = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "http://localhost:9878")
s3a_access_key = os.environ.get("S3A_ACCESS_KEY", "tazama")
s3a_secret_key = os.environ.get("S3A_SECRET_KEY", "tazama")
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .appName("Tazama-Dashboard")
    .master("local[*]")
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", s3a_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory", driver_memory)
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")

26/04/17 09:02:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 09:02:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Version: 3.4.2


In [3]:
# Import PySpark SQL functions and libraries
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✓ All required libraries imported successfully")

✓ All required libraries imported successfully


In [4]:
WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")
gold_alerts_path       = f"{WAREHOUSE_ROOT}/gold/alerts"
cases_gold_path   = f"{WAREHOUSE_ROOT}/gold/cases"
tasks_gold_path   = f"{WAREHOUSE_ROOT}/gold/tasks"
transactions_gold_path = f"{WAREHOUSE_ROOT}/gold/transactions"
nmap_gold_path = f"{WAREHOUSE_ROOT}/gold/network_map"
rules_gold_path   = f"{WAREHOUSE_ROOT}/gold/rules"
conditions_gold_path   = f"{WAREHOUSE_ROOT}/gold/conditions"

In [5]:
alerts=spark.read.format("hudi").load(gold_alerts_path)
cases=spark.read.format("hudi").load(cases_gold_path)
tasks=spark.read.format("hudi").load(tasks_gold_path)
transactions=spark.read.format("hudi").load(transactions_gold_path)
network_map=spark.read.format("hudi").load(nmap_gold_path)
rules=spark.read.format("hudi").load(rules_gold_path)
#conditions=spark.read.format("hudi").load(conditions_gold_path)

26/04/17 09:02:33 WARN DFSPropertiesConfiguration: Cannot find HUDI_CONF_DIR, please set it as the dir of hudi-defaults.conf
26/04/17 09:02:33 WARN DFSPropertiesConfiguration: Properties file file:/etc/hudi/conf/hudi-defaults.conf not found. Ignoring to load props file


## 2. Define Date Range and Granularity Selection
Configure flexible date ranges and time granularities for analysis.

In [6]:
from datetime import datetime, timedelta
from ipywidgets import DatePicker, Dropdown, Button, Output, HBox, VBox

# Get min/max dates from cases data with error handling
try:
    # Try to get valid min/max dates, filtering out null/invalid timestamps
    valid_cases = cases.filter(F.col("case_created_ts").isNotNull())
    max_date_result = valid_cases.agg(F.max(F.col("case_created_ts").cast("date"))).collect()[0][0]
    min_date_result = valid_cases.agg(F.min(F.col("case_created_ts").cast("date"))).collect()[0][0]
    
    if max_date_result and min_date_result:
        max_date = max_date_result
        min_date = min_date_result
    else:
        # Fallback if conversion fails
        max_date = datetime.today().date()
        min_date = (datetime.today() - timedelta(days=365)).date()
except:
    # Fallback to current date range if there's any error
    print("⚠️  Warning: Could not extract valid dates from case_created_ts. Using current date range.")
    max_date = datetime.today().date()
    min_date = (datetime.today() - timedelta(days=365)).date()

print(f"Data Range: {min_date} to {max_date}")

# ============================================================================
# PRE-CONFIGURED PERIODS
# ============================================================================
def get_period_dates(period_name):
    """Returns (start_date, end_date) for pre-configured periods"""
    end_date = max_date
    
    if period_name == "Last 30 days":
        start_date = end_date - timedelta(days=30)
    elif period_name == "Last 90 days":
        start_date = end_date - timedelta(days=90)
    elif period_name == "Last year":
        start_date = end_date - timedelta(days=365)
    elif period_name == "All history":
        start_date = min_date
    else:
        start_date = end_date - timedelta(days=30)
    
    return start_date, end_date

def get_previous_period(start_date, end_date):
    """Calculate previous period for trend comparison"""
    period_length = (end_date - start_date).days
    prev_end_date = start_date - timedelta(days=1)
    prev_start_date = prev_end_date - timedelta(days=period_length)
    return prev_start_date, prev_end_date

# ============================================================================
# TIME GRANULARITY OPTIONS
# ============================================================================
granularities = ["Daily", "Weekly", "Monthly", "Quarterly"]

def get_time_bucket(ts_col, granularity):
    """Create time bucket column based on granularity"""
    if granularity == "Daily":
        return F.to_date(ts_col)
    elif granularity == "Weekly":
        return F.date_trunc("week", ts_col)
    elif granularity == "Monthly":
        return F.date_trunc("month", ts_col)
    elif granularity == "Quarterly":
        return F.date_trunc("quarter", ts_col)
    return F.to_date(ts_col)

# ============================================================================
# INTERACTIVE CONTROLS
# ============================================================================
print("\n" + "="*80)
print("INTERACTIVE CONTROLS FOR ANALYSIS")
print("="*80)

# Selected parameters (can be modified)
selected_period = "Last 90 days"
selected_granularity = "Monthly"
selected_typologies = ["FRAUD", "AML", "FRAUD_AND_AML"]
value_threshold = 10000  # Configurable financial threshold

start_date, end_date = get_period_dates(selected_period)
prev_start_date, prev_end_date = get_previous_period(start_date, end_date)

print(f"\n✓ Selected Period: {selected_period}")
print(f"  Current Period: {start_date} to {end_date}")
print(f"  Previous Period: {prev_start_date} to {prev_end_date}")
print(f"✓ Time Granularity: {selected_granularity}")
print(f"✓ Selected Typologies: {', '.join(selected_typologies)}")
print(f"✓ High-Value Case Threshold: ${value_threshold:,.0f}")

Data Range: 2026-04-11 to 2026-04-13

INTERACTIVE CONTROLS FOR ANALYSIS

✓ Selected Period: Last 90 days
  Current Period: 2026-01-13 to 2026-04-13
  Previous Period: 2025-10-14 to 2026-01-12
✓ Time Granularity: Monthly
✓ Selected Typologies: FRAUD, AML, FRAUD_AND_AML
✓ High-Value Case Threshold: $10,000


## 3. Calculate Case Inventory Metrics
Compute metrics for total open cases, new cases (inflow), and closed cases (outflow) with period-over-period comparison.

In [7]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window

In [8]:
def calculate_case_inventory_metrics(start_date, end_date, prev_start_date, prev_end_date, alerts_df, cases_df, transactions_df):
    """Calculate case inventory metrics with trend comparison"""
    
    # ====================== CURRENT PERIOD ========================
    
    # 1. Total Open Cases
    total_open_cases = cases_df.filter(
        ~F.col("status").like("%CLOSED%") & 
        ~F.col("status").like("%COMPLETE%") & 
        ~F.col("status").like("%DISPOSED%")
    ).count()
    
    # Cases by status
    cases_by_status = cases_df.groupBy("status").agg(
        F.count("case_id").alias("count")
    ).orderBy(F.desc("count")).toPandas()
    
    # Cases by type (join with alerts)
    cases_with_type = cases_df.join(
        alerts_df.select("case_id", "alert_type_norm").distinct(),
        "case_id", "left"
    )
    
    aml_cases = cases_with_type.filter(F.col("alert_type_norm") == "AML").select("case_id").distinct().count()
    fraud_cases = cases_with_type.filter(F.col("alert_type_norm") == "FRAUD").select("case_id").distinct().count()
    fraud_aml_cases = cases_with_type.filter(F.col("alert_type_norm") == "FRAUD_AND_AML").select("case_id").distinct().count()
    
    # 2. New Cases (Inflow) - in the period
    cases_from_alerts = cases_df.filter(
        (F.col("case_creation_type") == "AUTOMATIC_SYSTEM") &
        (F.col("case_created_ts").cast("date") >= F.lit(start_date)) &
        (F.col("case_created_ts").cast("date") <= F.lit(end_date))
    ).count()
    
    manual_cases_created = cases_df.filter(
        (F.col("case_creation_type") != "AUTOMATIC_SYSTEM") &
        (F.col("case_created_ts").cast("date") >= F.lit(start_date)) &
        (F.col("case_created_ts").cast("date") <= F.lit(end_date))
    ).count()
    
    # 3. Closed Cases (Outflow)
    cases_closed_period = cases_df.filter(
        (F.col("case_updated_ts").cast("date") >= F.lit(start_date)) &
        (F.col("case_updated_ts").cast("date") <= F.lit(end_date)) &
        (F.col("status").like("%CLOSED%") | F.col("status").like("%COMPLETE%") | F.col("status").like("%DISPOSED%"))
    ).count()
    
    # 4. Auto-closed cases
    alerts_with_tx = alerts_df.join(
        transactions_df.select(F.col("tx_msg_id"), F.col("tx_status").alias("transaction_status")),
        "tx_msg_id", "left"
    )
    
    auto_closed_case_ids = alerts_with_tx.filter(
        (F.col("prediction_outcome_norm") == "FALSE_POSITIVE") |
        (
            (F.col("alert_type_norm") == "FRAUD") &
            (F.col("prediction_outcome_norm") == "TRUE_POSITIVE") &
            (F.col("transaction_status") == "BLOCKED")
        )
    ).select("case_id").distinct()
    
    auto_closed_period = cases_df.join(auto_closed_case_ids, "case_id", "inner").filter(
        (F.col("case_updated_ts").cast("date") >= F.lit(start_date)) &
        (F.col("case_updated_ts").cast("date") <= F.lit(end_date))
    ).count()
    
    # 5. Auto-close Reopen Rate
    auto_close_reopened = cases_df.join(auto_closed_case_ids, "case_id", "inner").filter(
        F.col("has_parent_case") == 1
    ).count()
    
    total_auto_closed = auto_closed_case_ids.count()
    auto_closure_reopen_rate = (auto_close_reopened / total_auto_closed * 100) if total_auto_closed > 0 else 0
    
    # 6. Reopened Case Rate
    cases_reopened = cases_df.filter(F.col("has_parent_case") == 1).count()
    cases_closed_total = cases_df.filter(
        F.col("status").like("%CLOSED%") | F.col("status").like("%COMPLETE%") | F.col("status").like("%DISPOSED%")
    ).count()
    reopened_case_rate = (cases_reopened / cases_closed_total * 100) if cases_closed_total > 0 else 0
    
    # ====================== PREVIOUS PERIOD (for trend) ========================
    
    cases_from_alerts_prev = cases_df.filter(
        (F.col("case_creation_type") == "AUTOMATIC_SYSTEM") &
        (F.col("case_created_ts").cast("date") >= F.lit(prev_start_date)) &
        (F.col("case_created_ts").cast("date") <= F.lit(prev_end_date))
    ).count()
    
    manual_cases_prev = cases_df.filter(
        (F.col("case_creation_type") != "AUTOMATIC_SYSTEM") &
        (F.col("case_created_ts").cast("date") >= F.lit(prev_start_date)) &
        (F.col("case_created_ts").cast("date") <= F.lit(prev_end_date))
    ).count()
    
    cases_closed_period_prev = cases_df.filter(
        (F.col("case_updated_ts").cast("date") >= F.lit(prev_start_date)) &
        (F.col("case_updated_ts").cast("date") <= F.lit(prev_end_date)) &
        (F.col("status").like("%CLOSED%") | F.col("status").like("%COMPLETE%") | F.col("status").like("%DISPOSED%"))
    ).count()
    
    return {
        "current": {
            "total_open_cases": total_open_cases,
            "cases_by_status": cases_by_status,
            "aml_cases": aml_cases,
            "fraud_cases": fraud_cases,
            "fraud_aml_cases": fraud_aml_cases,
            "cases_from_alerts": cases_from_alerts,
            "manual_cases": manual_cases_created,
            "auto_closed_cases": auto_closed_period,
            "auto_closure_reopen_rate": auto_closure_reopen_rate,
            "cases_closed": cases_closed_period,
            "reopened_case_rate": reopened_case_rate
        },
        "previous": {
            "cases_from_alerts": cases_from_alerts_prev,
            "manual_cases": manual_cases_prev,
            "cases_closed": cases_closed_period_prev
        }
    }

def get_trend_indicator(current, previous):
    """Return trend indicator (↑, ↓, →)"""
    if previous == 0:
        if current > 0:
            return "↑"
        return "→"
    change = ((current - previous) / previous * 100)
    if change > 5:
        return "↑"
    elif change < -5:
        return "↓"
    else:
        return "→"

# Calculate Case Inventory Metrics
print("\n" + "="*80)
print("CASE INVENTORY METRICS")
print("="*80)

metrics = calculate_case_inventory_metrics(start_date, end_date, prev_start_date, prev_end_date, alerts, cases, transactions)

print("\nA. TOTAL OPEN CASES")
print(f"  Total: {metrics['current']['total_open_cases']}")
print(f"  - AML Cases: {metrics['current']['aml_cases']}")
print(f"  - Fraud Cases: {metrics['current']['fraud_cases']}")
print(f"  - Fraud+AML Cases: {metrics['current']['fraud_aml_cases']}")

print("\nB. NEW CASES (Inflow)")
trend_alerts = get_trend_indicator(metrics['current']['cases_from_alerts'], metrics['previous']['cases_from_alerts'])
trend_manual = get_trend_indicator(metrics['current']['manual_cases'], metrics['previous']['manual_cases'])
print(f"  Alert-triggered Cases: {metrics['current']['cases_from_alerts']} {trend_alerts}")
print(f"  Manually Created Cases: {metrics['current']['manual_cases']} {trend_manual}")

print("\nC. CLOSED CASES (Outflow)")
trend_closed = get_trend_indicator(metrics['current']['cases_closed'], metrics['previous']['cases_closed'])
print(f"  Auto-closed Cases: {metrics['current']['auto_closed_cases']}")
print(f"  Auto-Closure Reopen Rate: {metrics['current']['auto_closure_reopen_rate']:.1f}%")
print(f"  Total Closed Cases: {metrics['current']['cases_closed']} {trend_closed}")
print(f"  Reopened Case Rate: {metrics['current']['reopened_case_rate']:.1f}%")


CASE INVENTORY METRICS

A. TOTAL OPEN CASES
  Total: 128381
  - AML Cases: 0
  - Fraud Cases: 0
  - Fraud+AML Cases: 0

B. NEW CASES (Inflow)
  Alert-triggered Cases: 128381 ↑
  Manually Created Cases: 0 →

C. CLOSED CASES (Outflow)
  Auto-closed Cases: 32095
  Auto-Closure Reopen Rate: 0.0%
  Total Closed Cases: 0 →
  Reopened Case Rate: 0.0%


## 4. Calculate Task and Investigation Metrics
Count open tasks by status and type, calculate investigations created/closed in period.

In [9]:
def calculate_task_and_investigation_metrics(start_date, end_date, tasks_df, cases_df):
    """Calculate task and investigation metrics"""
    
    # 1. Open Tasks
    open_tasks = tasks_df.filter(F.col("is_completed") == 0).count()
    
    # Tasks by status
    tasks_by_status = tasks_df.groupBy("status").agg(
        F.count("task_id").alias("count")
    ).orderBy(F.desc("count")).toPandas()
    
    # 2. Open tasks by type
    open_tasks_by_type = tasks_df.filter(F.col("is_completed") == 0).groupBy("candidate_group").agg(
        F.count("task_id").alias("count")
    ).orderBy(F.desc("count")).toPandas()
    
    # 3. Average open tasks per user
    avg_tasks_per_user = open_tasks / (
        tasks_df.filter(F.col("is_completed") == 0).select("assigned_user_id").distinct().count() 
        or 1
    )
    
    # 4. Investigation tasks by status
    investigation_tasks_by_status = tasks_df.filter(
        F.col("status") == "STATUS_20_IN_PROGRESS"
    ).groupBy("status").agg(
        F.count("task_id").alias("count")
    ).toPandas()
    
    # 5. Investigations created in period
    investigations_created = cases_df.join(
        tasks_df.filter(
            (F.col("task_created_ts").cast("date") >= F.lit(start_date)) &
            (F.col("task_created_ts").cast("date") <= F.lit(end_date))
        ).select("case_id").distinct(),
        "case_id", "inner"
    ).select("case_id").distinct().count()
    
    # 6. Investigations closed in period
    investigations_closed = cases_df.join(
        tasks_df.filter(
            (F.col("is_completed") == 1) &
            (F.col("task_completed_ts").cast("date") >= F.lit(start_date)) &
            (F.col("task_completed_ts").cast("date") <= F.lit(end_date))
        ).select("case_id").distinct(),
        "case_id", "inner"
    ).select("case_id").distinct().count()
    
    return {
        "open_tasks": open_tasks,
        "tasks_by_status": tasks_by_status,
        "open_tasks_by_type": open_tasks_by_type,
        "avg_tasks_per_user": avg_tasks_per_user,
        "investigation_tasks_by_status": investigation_tasks_by_status,
        "investigations_created": investigations_created,
        "investigations_closed": investigations_closed,
        "sar_str_filed": "N/A - Field not available in current schema"
    }

# Calculate Task and Investigation Metrics
print("\n" + "="*80)
print("TASK AND INVESTIGATION METRICS")
print("="*80)

task_metrics = calculate_task_and_investigation_metrics(start_date, end_date, tasks, cases)

print(f"\nA. OPEN TASKS")
print(f"  Total Open Tasks: {task_metrics['open_tasks']}")
print(f"  Average Tasks per User: {task_metrics['avg_tasks_per_user']:.1f}")

print(f"\nB. INVESTIGATIONS")
print(f"  Investigations Created: {task_metrics['investigations_created']}")
print(f"  Investigations Closed: {task_metrics['investigations_closed']}")
print(f"  SAR/STR Filed: {task_metrics['sar_str_filed']}")

print(f"\nC. OPEN TASKS BY TYPE (Top 5)")
print(task_metrics['open_tasks_by_type'].head(5).to_string(index=False))


TASK AND INVESTIGATION METRICS

A. OPEN TASKS
  Total Open Tasks: 128378
  Average Tasks per User: 128378.0

B. INVESTIGATIONS
  Investigations Created: 128378
  Investigations Closed: 0
  SAR/STR Filed: N/A - Field not available in current schema

C. OPEN TASKS BY TYPE (Top 5)
candidate_group  count
 INVESTIGATIONS 128378


## 5. Calculate Case Age and Timeliness Metrics
Generate case age distribution, MTTD, and alert-to-disposition time metrics.

In [10]:
def calculate_age_and_timeliness_metrics(cases_df, alerts_df, today_date=None):
    """Calculate case age distribution and timeliness metrics"""
    
    if today_date is None:
        today_date = datetime.today().date()
    
    # 1. Case Age Distribution - All open cases
    open_cases = cases_df.filter(
        ~F.col("status").like("%CLOSED%") & 
        ~F.col("status").like("%COMPLETE%") & 
        ~F.col("status").like("%DISPOSED%")
    ).withColumn(
        "age_days", 
        F.datediff(F.lit(today_date), F.col("case_created_ts").cast("date"))
    )
    
    # Define age buckets
    age_dist = open_cases.withColumn(
        "age_bucket",
        F.when(F.col("age_days") <= 7, "0-7 days (new)")
        .when(F.col("age_days") <= 14, "8-14 days (active)")
        .when(F.col("age_days") <= 30, "15-30 days (maturing)")
        .when(F.col("age_days") <= 60, "31-60 days (aged)")
        .otherwise("60+ days (critical)")
    ).groupBy("age_bucket").agg(
        F.count("case_id").alias("count")
    ).orderBy("age_bucket").toPandas()
    
    # 2. MTTD - Mean Time to Disposition
    closed_cases = cases_df.filter(
        (F.col("created_to_updated_ms") > 0) &
        (F.col("status").isin(
            "STATUS_81_CLOSED_REFUTED",
            "STATUS_82_CLOSED_CONFIRMED",
            "STATUS_83_CLOSED_INCONCLUSIVE"
        ))
    )
    
    avg_mttd_ms = closed_cases.agg(F.avg("created_to_updated_ms")).collect()[0][0]
    mttd_hours = (avg_mttd_ms / (1000 * 60 * 60)) if avg_mttd_ms else 0
    mttd_days = mttd_hours / 24
    
    # 3. MTTD by disposition
    mttd_by_disposition = closed_cases.groupBy("status").agg(
        F.avg("created_to_updated_ms").alias("avg_ms"),
        F.count("case_id").alias("count")
    ).withColumn(
        "avg_hours",
        F.round(F.col("avg_ms") / (1000 * 60 * 60), 2)
    ).withColumn(
        "avg_days",
        F.round(F.col("avg_hours") / 24, 2)
    ).orderBy("status").toPandas()
    
    # 4. MTTD by case type (Fraud/AML)
    cases_with_type = cases_df.join(
        alerts_df.select("case_id", "alert_type_norm").distinct(),
        "case_id", "left"
    ).filter(
        (F.col("created_to_updated_ms") > 0) &
        (F.col("status").isin(
            "STATUS_81_CLOSED_REFUTED",
            "STATUS_82_CLOSED_CONFIRMED",
            "STATUS_83_CLOSED_INCONCLUSIVE"
        ))
    )
    
    mttd_by_type = cases_with_type.groupBy("alert_type_norm").agg(
        F.avg("created_to_updated_ms").alias("avg_ms"),
        F.count("case_id").alias("count")
    ).withColumn(
        "avg_hours",
        F.round(F.col("avg_ms") / (1000 * 60 * 60), 2)
    ).orderBy("alert_type_norm").toPandas()
    
    return {
        "age_distribution": age_dist,
        "mttd_days": mttd_days,
        "mttd_hours": mttd_hours,
        "mttd_by_disposition": mttd_by_disposition,
        "mttd_by_type": mttd_by_type
    }

# Calculate Age and Timeliness Metrics
print("\n" + "="*80)
print("CASE AGE & TIMELINESS METRICS")
print("="*80)

timeliness_metrics = calculate_age_and_timeliness_metrics(cases, alerts)

print("\nA. CASE AGE DISTRIBUTION (Open Cases)")
print(timeliness_metrics['age_distribution'].to_string(index=False))

print(f"\nB. MEAN TIME TO DISPOSITION (MTTD)")
print(f"  Average: {timeliness_metrics['mttd_days']:.1f} days ({timeliness_metrics['mttd_hours']:.1f} hours)")

print(f"\nC. MTTD BY DISPOSITION")
print(timeliness_metrics['mttd_by_disposition'][['status', 'count', 'avg_days']].to_string(index=False))

print(f"\nD. MTTD BY CASE TYPE")
print(timeliness_metrics['mttd_by_type'][['alert_type_norm', 'count', 'avg_hours']].to_string(index=False))


CASE AGE & TIMELINESS METRICS

A. CASE AGE DISTRIBUTION (Open Cases)
    age_bucket  count
0-7 days (new) 128381

B. MEAN TIME TO DISPOSITION (MTTD)
  Average: 0.0 days (0.0 hours)

C. MTTD BY DISPOSITION
Empty DataFrame
Columns: [status, count, avg_days]
Index: []

D. MTTD BY CASE TYPE
Empty DataFrame
Columns: [alert_type_norm, count, avg_hours]
Index: []


## 6. Calculate High-Value Open Case Tracking Metrics
Identify cases above financial threshold with concentration analysis and drill-down.

In [11]:
def calculate_high_value_metrics(cases_df, transactions_df, alerts_df, threshold=10000):
    """Calculate high-value case tracking metrics"""
    
    # Join cases -> alerts -> transactions to get transaction amounts
    # First join cases with alerts to get tx_msg_id
    cases_with_alerts = cases_df.join(
        alerts_df.select("case_id", "tx_msg_id"),
        "case_id",
        "left"
    )
    
    # Then join with transactions using tx_msg_id
    cases_with_value = cases_with_alerts.join(
        transactions_df.select("tx_msg_id", "tx_amount"),
        "tx_msg_id",
        "left"
    ).withColumn("tx_amount", F.coalesce(F.col("tx_amount"), F.lit(0)))
    
    # Filter open cases
    open_high_value = cases_with_value.filter(
        (~F.col("status").like("%CLOSED%") & 
         ~F.col("status").like("%COMPLETE%") & 
         ~F.col("status").like("%DISPOSED%")) &
        (F.col("tx_amount") >= threshold)
    )
    
    # Calculate metrics
    high_value_count = open_high_value.count()
    total_value_at_risk = open_high_value.agg(F.sum("tx_amount")).collect()[0][0] or 0
    avg_value_per_case = (total_value_at_risk / high_value_count) if high_value_count > 0 else 0
    
    # Top 10 cases by value
    top_10_cases = open_high_value.groupBy("case_id").agg(
        F.max("tx_amount").alias("value"),
        F.first("status").alias("status"),
        F.first("case_created_ts").cast("string").alias("created_ts")
    ).orderBy(F.desc("value")).limit(10).toPandas()
    
    return {
        "high_value_count": high_value_count,
        "total_value_at_risk": total_value_at_risk,
        "avg_value_per_case": avg_value_per_case,
        "top_10_cases": top_10_cases
    }

# Calculate High-Value Metrics
print("\n" + "="*80)
print("HIGH-VALUE OPEN CASE TRACKING")
print("="*80)

hv_metrics = calculate_high_value_metrics(cases, transactions, alerts, threshold=value_threshold)

print(f"\nA. HIGH-VALUE CASES (Above ${value_threshold:,.0f})")
print(f"  Count: {hv_metrics['high_value_count']}")
print(f"  Total Value at Risk: ${hv_metrics['total_value_at_risk']:,.2f}")
print(f"  Average Value per Case: ${hv_metrics['avg_value_per_case']:,.2f}")

print(f"\nB. TOP 10 CASES BY VALUE")
print(hv_metrics['top_10_cases'][['case_id', 'value', 'status']].head(10).to_string(index=False))


HIGH-VALUE OPEN CASE TRACKING

A. HIGH-VALUE CASES (Above $10,000)
  Count: 0
  Total Value at Risk: $0.00
  Average Value per Case: $0.00

B. TOP 10 CASES BY VALUE
Empty DataFrame
Columns: [case_id, value, status]
Index: []


## 7. Calculate Case Status and Workflow Metrics
Compute case distribution by status and disposition outcomes.

In [12]:
import pandas as pd

In [13]:
def calculate_status_and_workflow_metrics(cases_df):
    """Calculate case status and disposition metrics"""
    
    # 1. Cases by status
    cases_by_status = cases_df.groupBy("status").agg(
        F.count("case_id").alias("count")
    ).withColumn(
        "percentage",
        F.round((F.col("count") / F.sum("count").over(Window.partitionBy())) * 100, 2)
    ).orderBy(F.desc("count")).toPandas()
    
    # 2. Disposition Distribution for closed cases
    closed_cases = cases_df.filter(
        F.col("status").like("%CLOSED%") | 
        F.col("status").like("%COMPLETE%") | 
        F.col("status").like("%DISPOSED%")
    )
    
    total_closed = closed_cases.count()
    
    confirmed_cases = closed_cases.filter(
        F.col("status") == "STATUS_82_CLOSED_CONFIRMED"
    ).count()
    
    refuted_cases = closed_cases.filter(
        F.col("status") == "STATUS_81_CLOSED_REFUTED"
    ).count()
    
    inconclusive_cases = closed_cases.filter(
        F.col("status") == "STATUS_83_CLOSED_INCONCLUSIVE"
    ).count()
    
    disposition_pct = {
        "Confirmed": (confirmed_cases / total_closed * 100) if total_closed > 0 else 0,
        "Refuted": (refuted_cases / total_closed * 100) if total_closed > 0 else 0,
        "Inconclusive": (inconclusive_cases / total_closed * 100) if total_closed > 0 else 0
    }
    
    disposition_df = pd.DataFrame({
        "Disposition": list(disposition_pct.keys()),
        "Count": [confirmed_cases, refuted_cases, inconclusive_cases],
        "Percentage": list(disposition_pct.values())
    })
    
    return {
        "cases_by_status": cases_by_status,
        "total_closed": total_closed,
        "disposition_distribution": disposition_df
    }

# Calculate Status and Workflow Metrics
print("\n" + "="*80)
print("CASE STATUS & WORKFLOW METRICS")
print("="*80)

status_metrics = calculate_status_and_workflow_metrics(cases)

print(f"\nA. CASES BY WORKFLOW STATUS")
print(status_metrics['cases_by_status'][['status', 'count', 'percentage']].to_string(index=False))

print(f"\nB. DISPOSITION DISTRIBUTION (Closed Cases)")
print(f"  Total Closed Cases: {status_metrics['total_closed']}")
print(status_metrics['disposition_distribution'].to_string(index=False))


CASE STATUS & WORKFLOW METRICS


26/04/17 09:02:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 09:02:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 09:02:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 09:02:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 09:02:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 09:02:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.



A. CASES BY WORKFLOW STATUS
         status  count  percentage
STATUS_00_DRAFT 128381       100.0

B. DISPOSITION DISTRIBUTION (Closed Cases)
  Total Closed Cases: 0
 Disposition  Count  Percentage
   Confirmed      0           0
     Refuted      0           0
Inconclusive      0           0


## 8. Visualize Metrics with Trend Indicators
Create interactive visualizations for all KPIs with current values and trend indicators.

In [14]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def create_kpi_card(title, current_value, previous_value, unit="", format_str=",.0f"):
    """Create a KPI card visualization with trend indicator"""
    trend = get_trend_indicator(current_value, previous_value)
    trend_color = "green" if trend == "↑" else ("red" if trend == "↓" else "gray")
    
    fig = go.Figure(go.Indicator(
        mode="number",
        value=current_value,
        title={"text": title},
        number={"suffix": unit, "valueformat": format_str},
        domain={"x": [0, 1], "y": [0, 1]},
        gauge={"axis": {"visible": False}},
    ))
    
    fig.add_annotation(
        text=f"<b>{trend}</b>",
        x=0.9, y=0.1,
        showarrow=False,
        font={"size": 24, "color": trend_color}
    )
    
    fig.update_layout(height=300, font={"size": 16})
    return fig

# Create KPI visualizations
print("\n" + "="*80)
print("KPI VISUALIZATIONS")
print("="*80)

# Row 1: Case Inventory KPIs
fig_open_cases = create_kpi_card(
    "Total Open Cases",
    metrics['current']['total_open_cases'],
    0  # No previous period baseline
)
fig_open_cases.show()

fig_inflow = create_kpi_card(
    "Cases from Alerts (Period)",
    metrics['current']['cases_from_alerts'],
    metrics['previous']['cases_from_alerts']
)
fig_inflow.show()

fig_outflow = create_kpi_card(
    "Cases Closed (Period)",
    metrics['current']['cases_closed'],
    metrics['previous']['cases_closed']
)
fig_outflow.show()

# Case Age Distribution Histogram
fig_age = go.Figure()
age_data = timeliness_metrics['age_distribution']
fig_age.add_trace(go.Bar(
    x=age_data['age_bucket'],
    y=age_data['count'],
    marker={"color": ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]},
    text=age_data['count'],
    textposition='outside'
))
fig_age.update_layout(
    title="Case Age Distribution (Open Cases)",
    xaxis_title="Age Bucket",
    yaxis_title="Number of Cases",
    height=400,
    showlegend=False
)
fig_age.show()

# Cases by Status Distribution
fig_status = go.Figure()
status_data = status_metrics['cases_by_status'].head(8)
fig_status.add_trace(go.Bar(
    x=status_data['status'],
    y=status_data['count'],
    marker={"color": status_data['count'], "colorscale": "Viridis"},
    text=status_data['count'],
    textposition='outside'
))
fig_status.update_layout(
    title="Cases by Workflow Status",
    xaxis_title="Status",
    yaxis_title="Count",
    height=400,
    showlegend=False
)
fig_status.update_xaxes(tickangle=-45)
fig_status.show()

# Disposition Distribution Pie Chart
fig_disposition = go.Figure()
disposition_data = status_metrics['disposition_distribution']
fig_disposition.add_trace(go.Pie(
    labels=disposition_data['Disposition'],
    values=disposition_data['Count'],
    textposition='inside',
    textinfo='label+percent'
))
fig_disposition.update_layout(
    title="Closed Cases by Disposition",
    height=400
)
fig_disposition.show()

print("\n✓ Visualizations created successfully!")


KPI VISUALIZATIONS



✓ Visualizations created successfully!


## 9. Compare Multiple Typologies Side-by-Side
Filter metrics by fraud typologies and enable dynamic comparison.

In [15]:
def get_typology_metrics(typology_name, start_date, end_date, alerts_df, cases_df, transactions_df):
    """Calculate metrics for a specific fraud typology"""
    
    # Filter alerts by typology
    typology_alerts = alerts_df.filter(F.col("alert_type_norm") == typology_name)
    typology_case_ids = typology_alerts.select("case_id").distinct()
    
    # Filter cases for this typology
    typology_cases = cases_df.join(typology_case_ids, "case_id", "inner")
    
    # Calculate metrics
    total_cases = typology_cases.count()
    
    open_cases = typology_cases.filter(
        ~F.col("status").like("%CLOSED%") & 
        ~F.col("status").like("%COMPLETE%") & 
        ~F.col("status").like("%DISPOSED%")
    ).count()
    
    closed_cases = typology_cases.filter(
        F.col("status").like("%CLOSED%") | 
        F.col("status").like("%COMPLETE%") | 
        F.col("status").like("%DISPOSED%")
    ).count()
    
    # Alert count
    alert_count = typology_alerts.select("alert_id").distinct().count()
    
    # Calculate MTTD
    closed_typology_cases = typology_cases.filter(
        (F.col("created_to_updated_ms") > 0) &
        (
            F.col("status").like("%CLOSED%") | 
            F.col("status").like("%COMPLETE%") | 
            F.col("status").like("%DISPOSED%")
        )
    )
    
    avg_mttd_ms = closed_typology_cases.agg(F.avg("created_to_updated_ms")).collect()[0][0]
    mttd_hours = (avg_mttd_ms / (1000 * 60 * 60)) if avg_mttd_ms else 0
    
    # Confirmation rate
    confirmed = typology_cases.filter(
        F.col("status") == "STATUS_82_CLOSED_CONFIRMED"
    ).count()
    
    confirmation_rate = (confirmed / closed_cases * 100) if closed_cases > 0 else 0
    
    # False positive rate
    false_positives = typology_alerts.filter(
        F.col("prediction_outcome_norm") == "FALSE_POSITIVE"
    ).count()
    
    false_positive_rate = (false_positives / alert_count * 100) if alert_count > 0 else 0
    
    return {
        "typology": typology_name,
        "total_cases": total_cases,
        "open_cases": open_cases,
        "closed_cases": closed_cases,
        "alert_count": alert_count,
        "mttd_hours": mttd_hours,
        "confirmation_rate": confirmation_rate,
        "false_positive_rate": false_positive_rate
    }


# ============================
# MAIN EXECUTION
# ============================

# Get available typologies
available_typologies = alerts.select("alert_type_norm").distinct().filter(
    F.col("alert_type_norm").isNotNull()
).rdd.flatMap(lambda x: x).collect()

print("\n" + "="*80)
print("TYPOLOGY COMPARISON")
print("="*80)
print(f"\nAvailable Typologies: {available_typologies}")

# Calculate metrics
typology_metrics_list = []

for typology in available_typologies:
    try:
        metrics_dict = get_typology_metrics(
            typology, start_date, end_date, alerts, cases, transactions
        )
        typology_metrics_list.append(metrics_dict)
    except Exception as e:
        print(f"Error processing {typology}: {str(e)}")


# ============================
# CREATE DATAFRAME
# ============================

typology_comparison = pd.DataFrame(typology_metrics_list)

print("\nTYPOLOGY COMPARISON TABLE")

if typology_comparison.empty:
    print("No data available.")
else:
    print(typology_comparison.to_string(index=False))


# ============================
# VISUALIZATION
# ============================

if not typology_comparison.empty and "typology" in typology_comparison.columns:
    
    fig_typology_comp = go.Figure()

    metrics_to_plot = [
        'alert_count',
        'total_cases',
        'closed_cases',
        'confirmation_rate'
    ]

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    for i, metric in enumerate(metrics_to_plot):
        if metric in typology_comparison.columns:
            fig_typology_comp.add_trace(go.Bar(
                x=typology_comparison['typology'],
                y=typology_comparison[metric],
                name=metric.replace('_', ' ').title(),
                marker={'color': colors[i]}
            ))

    fig_typology_comp.update_layout(
        title="Typology Comparison - Key Metrics",
        xaxis_title="Typology",
        yaxis_title="Count / Rate",
        barmode='group',
        height=400
    )

    fig_typology_comp.show()

    print("\n✓ Typology comparison complete!")

else:
    print("\n⚠ Cannot plot — 'typology' column missing or no data.")


TYPOLOGY COMPARISON

Available Typologies: []

TYPOLOGY COMPARISON TABLE
No data available.

⚠ Cannot plot — 'typology' column missing or no data.


## 10. Rank and Sort Typologies by Selected KPI
Dynamically sort typologies by selected metrics with performance badges.

In [17]:
def rank_typologies(comparison_df, metric_name, ascending=False):
    """Rank typologies by selected metric and add performance badges"""
    
    # 🔒 Safety checks
    if comparison_df is None or comparison_df.empty:
        print(f"⚠ No data available for ranking ({metric_name})")
        return pd.DataFrame()

    # normalize column names (handles case/space issues)
    comparison_df.columns = comparison_df.columns.str.strip().str.lower()
    metric_name = metric_name.lower()

    if metric_name not in comparison_df.columns:
        print(f"⚠ Column '{metric_name}' missing")
        print("Available columns:", list(comparison_df.columns))
        return pd.DataFrame()

    if "typology" not in comparison_df.columns:
        print("⚠ Column 'typology' missing")
        print("Available columns:", list(comparison_df.columns))
        return pd.DataFrame()

    # ensure numeric
    comparison_df[metric_name] = pd.to_numeric(
        comparison_df[metric_name], errors='coerce'
    ).fillna(0)

    ranked_df = comparison_df.sort_values(
        by=metric_name, ascending=ascending
    ).reset_index(drop=True)

    ranked_df['Rank'] = range(1, len(ranked_df) + 1)

    # badges
    ranked_df['Performance'] = ranked_df['Rank'].apply(
        lambda r: "🥇 Top Performer" if r == 1 else
                  ("⚠️ Underperformer" if r == len(ranked_df) else "→ Standard")
    )

    return ranked_df[['Rank', 'typology', metric_name, 'Performance']]


# =========================================
# DEBUG (VERY IMPORTANT - KEEP THIS)
# =========================================
print("\nDEBUG CHECK")
print("Rows:", len(typology_comparison))
print("Columns:", list(typology_comparison.columns))
print(typology_comparison.head())


# =========================================
# RANKINGS
# =========================================

print("\n" + "="*80)
print("TYPOLOGY RANKING BY SELECTED METRICS")
print("="*80)

# Rank by Alert Count
print("\n1. RANKING BY ALERT COUNT (Highest)")
df1 = rank_typologies(typology_comparison.copy(), 'alert_count', ascending=False)
if not df1.empty:
    print(df1.to_string(index=False))

# Rank by Confirmation Rate
print("\n2. RANKING BY CONFIRMATION RATE (Highest)")
df2 = rank_typologies(typology_comparison.copy(), 'confirmation_rate', ascending=False)
if not df2.empty:
    print(df2.to_string(index=False))

# Rank by False Positive Rate
print("\n3. RANKING BY FALSE POSITIVE RATE (Lowest)")
df3 = rank_typologies(typology_comparison.copy(), 'false_positive_rate', ascending=True)
if not df3.empty:
    print(df3.to_string(index=False))

# Rank by MTTD
print("\n4. RANKING BY MEAN TIME TO DISPOSITION (Lowest)")
df4 = rank_typologies(typology_comparison.copy(), 'mttd_hours', ascending=True)
if not df4.empty:
    print(df4.to_string(index=False))


# =========================================
# VISUALIZATION (safe)
# =========================================

if not df2.empty:
    fig_ranked = go.Figure()
    fig_ranked.add_trace(go.Bar(
        x=df2['typology'],
        y=df2['confirmation_rate'],
        marker={'color': df2['Rank'], 'colorscale': 'RdYlGn'},
        text=[f"{v:.1f}%" for v in df2['confirmation_rate']],
        textposition='outside'
    ))
    fig_ranked.update_layout(
        title="Typologies Ranked by Confirmation Rate (%)",
        xaxis_title="Typology",
        yaxis_title="Confirmation Rate (%)",
        height=400
    )
    fig_ranked.show()

print("\n✓ Typology ranking complete!")


DEBUG CHECK
Rows: 0
Columns: []
Empty DataFrame
Columns: []
Index: []

TYPOLOGY RANKING BY SELECTED METRICS

1. RANKING BY ALERT COUNT (Highest)
⚠ No data available for ranking (alert_count)

2. RANKING BY CONFIRMATION RATE (Highest)
⚠ No data available for ranking (confirmation_rate)

3. RANKING BY FALSE POSITIVE RATE (Lowest)
⚠ No data available for ranking (false_positive_rate)

4. RANKING BY MEAN TIME TO DISPOSITION (Lowest)
⚠ No data available for ranking (mttd_hours)

✓ Typology ranking complete!


## 11. Export Dashboard Results
Generate HTML and summary exports for offline review and stakeholder sharing.

In [18]:
from datetime import datetime
import json

def generate_dashboard_summary(metrics, task_metrics, timeliness_metrics, hv_metrics, 
                              status_metrics, typology_comparison, selected_period, 
                              selected_granularity, value_threshold):
    """Generate a comprehensive dashboard summary"""
    
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    summary = {
        "dashboard_metadata": {
            "generated_at": timestamp,
            "selected_period": selected_period,
            "selected_granularity": selected_granularity,
            "high_value_threshold": f"${value_threshold:,.0f}"
        },
        "case_inventory_summary": {
            "total_open_cases": metrics['current']['total_open_cases'],
            "open_aml_cases": metrics['current']['aml_cases'],
            "open_fraud_cases": metrics['current']['fraud_cases'],
            "open_fraud_aml_cases": metrics['current']['fraud_aml_cases'],
            "new_cases_period": metrics['current']['cases_from_alerts'] + metrics['current']['manual_cases'],
            "closed_cases_period": metrics['current']['cases_closed'],
            "auto_closure_reopen_rate": round(metrics['current']['auto_closure_reopen_rate'], 2),
            "reopened_case_rate": round(metrics['current']['reopened_case_rate'], 2)
        },
        "task_summary": {
            "open_tasks": task_metrics['open_tasks'],
            "avg_tasks_per_user": round(task_metrics['avg_tasks_per_user'], 2),
            "investigations_created": task_metrics['investigations_created'],
            "investigations_closed": task_metrics['investigations_closed']
        },
        "timeliness_summary": {
            "mean_time_to_disposition_days": round(timeliness_metrics['mttd_days'], 2),
            "mean_time_to_disposition_hours": round(timeliness_metrics['mttd_hours'], 2)
        },
        "high_value_summary": {
            "high_value_cases_count": hv_metrics['high_value_count'],
            "total_value_at_risk": round(hv_metrics['total_value_at_risk'], 2),
            "avg_value_per_case": round(hv_metrics['avg_value_per_case'], 2)
        },
        "workflow_summary": {
            "total_closed_cases": status_metrics['total_closed'],
            "disposition": status_metrics['disposition_distribution'].set_index('Disposition').to_dict('index')
        },
        "typology_summary": typology_comparison.to_dict('records')
    }
    
    return summary

def export_dashboard_to_html(summary, metrics, timeliness_metrics, typology_comparison, 
                             status_metrics, filename="/tmp/fraud_typology_dashboard.html"):
    """Export dashboard to HTML format"""
    
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Fraud Typology Effectiveness Dashboard</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f5f5f5; }}
            .header {{ background-color: #1a1a1a; color: white; padding: 20px; border-radius: 5px; margin-bottom: 20px; }}
            .section {{ background-color: white; padding: 20px; margin-bottom: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
            .section h2 {{ color: #1a1a1a; border-bottom: 2px solid #4CAF50; padding-bottom: 10px; }}
            .metric-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin: 15px 0; }}
            .metric-card {{ background-color: #f9f9f9; padding: 15px; border-left: 4px solid #4CAF50; border-radius: 3px; }}
            .metric-label {{ font-size: 12px; color: #666; text-transform: uppercase; }}
            .metric-value {{ font-size: 24px; font-weight: bold; color: #1a1a1a; margin: 5px 0; }}
            table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
            th {{ background-color: #4CAF50; color: white; padding: 10px; text-align: left; }}
            td {{ padding: 10px; border-bottom: 1px solid #ddd; }}
            tr:hover {{ background-color: #f5f5f5; }}
            .footer {{ color: #666; font-size: 12px; margin-top: 30px; text-align: center; }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Fraud Typology Effectiveness Dashboard</h1>
            <p>Generated: {summary['dashboard_metadata']['generated_at']}</p>
            <p>Period: {summary['dashboard_metadata']['selected_period']} | Granularity: {summary['dashboard_metadata']['selected_granularity']}</p>
        </div>

        <div class="section">
            <h2>Case Inventory Metrics</h2>
            <div class="metric-grid">
                <div class="metric-card">
                    <div class="metric-label">Total Open Cases</div>
                    <div class="metric-value">{summary['case_inventory_summary']['total_open_cases']}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-label">AML Cases</div>
                    <div class="metric-value">{summary['case_inventory_summary']['open_aml_cases']}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-label">Fraud Cases</div>
                    <div class="metric-value">{summary['case_inventory_summary']['open_fraud_cases']}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-label">New Cases (Period)</div>
                    <div class="metric-value">{summary['case_inventory_summary']['new_cases_period']}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-label">Closed Cases (Period)</div>
                    <div class="metric-value">{summary['case_inventory_summary']['closed_cases_period']}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-label">Reopen Rate</div>
                    <div class="metric-value">{summary['case_inventory_summary']['reopened_case_rate']:.1f}%</div>
                </div>
            </div>
        </div>

        <div class="section">
            <h2>Timeliness Metrics</h2>
            <div class="metric-grid">
                <div class="metric-card">
                    <div class="metric-label">Mean Time to Disposition</div>
                    <div class="metric-value">{summary['timeliness_summary']['mean_time_to_disposition_days']:.1f} days</div>
                </div>
            </div>
            <h3>Case Age Distribution</h3>
            <table>
                <tr>
                    <th>Age Bucket</th>
                    <th>Count</th>
                </tr>
    """
    
    for _, row in timeliness_metrics['age_distribution'].iterrows():
        html_content += f"<tr><td>{row['age_bucket']}</td><td>{row['count']}</td></tr>"
    
    html_content += f"""
            </table>
        </div>

        <div class="section">
            <h2>High-Value Cases</h2>
            <div class="metric-grid">
                <div class="metric-card">
                    <div class="metric-label">High-Value Cases</div>
                    <div class="metric-value">{summary['high_value_summary']['high_value_cases_count']}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-label">Total Value at Risk</div>
                    <div class="metric-value">${summary['high_value_summary']['total_value_at_risk']:,.0f}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-label">Avg Value/Case</div>
                    <div class="metric-value">${summary['high_value_summary']['avg_value_per_case']:,.0f}</div>
                </div>
            </div>
        </div>

        <div class="section">
            <h2>Case Disposition</h2>
    """
    
    for _, row in status_metrics['disposition_distribution'].iterrows():
        html_content += f"<p>{row['Disposition']}: {row['Count']} ({row['Percentage']:.1f}%)</p>"
    
    html_content += f"""
        </div>

        <div class="section">
            <h2>Typology Comparison</h2>
            <table>
                <tr>
                    <th>Typology</th>
                    <th>Total Cases</th>
                    <th>Open Cases</th>
                    <th>Alert Count</th>
                    <th>Confirmation Rate</th>
                    <th>False Positive Rate</th>
                    <th>MTTD (hours)</th>
                </tr>
    """
    
    for _, row in typology_comparison.iterrows():
        html_content += f"""
        <tr>
            <td>{row['typology']}</td>
            <td>{row['total_cases']}</td>
            <td>{row['open_cases']}</td>
            <td>{row['alert_count']}</td>
            <td>{row['confirmation_rate']:.1f}%</td>
            <td>{row['false_positive_rate']:.1f}%</td>
            <td>{row['mttd_hours']:.1f}</td>
        </tr>
        """
    
    html_content += """
            </table>
        </div>

        <div class="footer">
            <p>This dashboard was automatically generated from Gold-layer data.</p>
            <p>For questions or feedback, contact the Fraud Analytics team.</p>
        </div>
    </body>
    </html>
    """
    
    with open(filename, 'w') as f:
        f.write(html_content)
    
    return filename

# Generate summary
print("\n" + "="*80)
print("EXPORT & SUMMARY")
print("="*80)

dashboard_summary = generate_dashboard_summary(
    metrics, task_metrics, timeliness_metrics, hv_metrics, status_metrics,
    typology_comparison, selected_period, selected_granularity, value_threshold
)

# Save as JSON
json_filename = "/tmp/fraud_typology_dashboard_summary.json"
with open(json_filename, 'w') as f:
    json.dump(dashboard_summary, f, indent=2)
print(f"\n✓ Summary exported to JSON: {json_filename}")

# Export to HTML
html_filename = export_dashboard_to_html(
    dashboard_summary, metrics, timeliness_metrics, typology_comparison, status_metrics
)
print(f"✓ Dashboard exported to HTML: {html_filename}")

# Display summary statistics
print("\n" + "="*80)
print("DASHBOARD SUMMARY STATISTICS")
print("="*80)
print(json.dumps(dashboard_summary, indent=2))

print("\n✓ Dashboard export complete!")


EXPORT & SUMMARY

✓ Summary exported to JSON: /tmp/fraud_typology_dashboard_summary.json
✓ Dashboard exported to HTML: /tmp/fraud_typology_dashboard.html

DASHBOARD SUMMARY STATISTICS
{
  "dashboard_metadata": {
    "generated_at": "2026-04-17 09:12:02",
    "selected_period": "Last 90 days",
    "selected_granularity": "Monthly",
    "high_value_threshold": "$10,000"
  },
  "case_inventory_summary": {
    "total_open_cases": 128381,
    "open_aml_cases": 0,
    "open_fraud_cases": 0,
    "open_fraud_aml_cases": 0,
    "new_cases_period": 128381,
    "closed_cases_period": 0,
    "auto_closure_reopen_rate": 0.0,
    "reopened_case_rate": 0
  },
  "task_summary": {
    "open_tasks": 128378,
    "avg_tasks_per_user": 128378.0,
    "investigations_created": 128378,
    "investigations_closed": 0
  },
  "timeliness_summary": {
    "mean_time_to_disposition_days": 0.0,
    "mean_time_to_disposition_hours": 0
  },
  "high_value_summary": {
    "high_value_cases_count": 0,
    "total_value_a